# 07 · Paper tables, figures and statistics from `results_v2` only  (CPU is fine, ~2–5 min)

Every number in the paper should come from this notebook. It reads only the result files written by
notebooks 00–06 and writes LaTeX/CSV tables and 300-dpi figures to `SSL_AgriStudy/paper_v2/`.
Missing inputs are skipped with a message.

**v2 changes:** label-efficiency x-axis in images per class with duplicate budgets removed (1 image/class
on Flowers-102/CUB is plotted and tested once); readable names for every checkpoint; significance tests for
label efficiency (paired over 20 subsets), fine-tuning and distillation (Welch over seeds), all Holm-corrected;
teacher accuracies; a log-linear scaling fit (indicative extrapolation); Pareto plot with every student and the teacher.

In [ ]:
%%writefile /content/sslcommon.py
"""
sslcommon.py — shared, validated code for the SSL_AgriStudy re-runs.

Every notebook writes this exact file and imports it, so all experiments use
ONE loader, ONE split definition, ONE linear-probe protocol and ONE set of
statistics. Fixes relative to the audited July runs:

  * Backbone loading is STRICT: no key filtering beyond the DINO projection
    head; any missing/unexpected backbone key raises an error. (The old
    Phase 2 / Phase 4 loaders dropped every 'mlp' key with strict=False.)
  * Splits: model selection on VAL, reporting on TEST. For Flowers102 and CUB
    the folder called 'val' is the official test set, so a stratified
    validation split is carved from 'train' (fixed seed, saved to JSON).
  * Linear probe: C chosen on val, refit on train+val, scored once on test.
  * Variance comes from things that actually vary (label subsets, fine-tune
    seeds); full-data linear probes get bootstrap CIs and paired tests.
"""
import os, json, time, random, shutil, hashlib, platform, subprocess, sys, copy
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# ───────────────────────── configuration ─────────────────────────
BASE      = os.environ.get('SSL_BASE', '/content/drive/MyDrive/SSL_AgriStudy')
LOCAL     = os.environ.get('SSL_LOCAL', '/content/ssl_data')
RESULTS   = f'{BASE}/results_v2'           # new, clean results folder
FEATS     = f'{BASE}/features_v2'          # new feature cache (never reuse old)
SPLITS    = f'{RESULTS}/splits'
CKPT_PRE  = f'{BASE}/checkpoints/pretrained'
CKPT_DOM  = f'{BASE}/checkpoints/domain_dino'      # 9 clean from-scratch runs
CKPT_DOM2 = f'{BASE}/checkpoints/domain_dino_v2'   # new runs from notebook 02
DINO_REPO = os.environ.get('SSL_DINO_REPO', '/content/dino_vanilla')

# dataset key -> folder under BASE/datasets
DATASETS = {
    'FloraBD':      'FloraBD',
    'RiceLeaf':     'RiceLeafDiseaseBD',
    'PlantVillage': 'PlantVillage',
    'Flowers102':   'Flowers102',
    'CUB':          'CUB',
}
# datasets whose 'val' folder is really the official TEST set
VAL_IS_TEST = {'Flowers102', 'CUB'}
CARVE_VAL_FRAC = 0.20     # fraction of train per class held out as val there
SPLIT_SEED = 0

IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = min(8, os.cpu_count() or 2)

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
EVAL_TF = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
TRAIN_TF = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_dirs():
    for d in (RESULTS, FEATS, SPLITS, CKPT_DOM2):
        os.makedirs(d, exist_ok=True)


# ───────────────────────── local data copy ─────────────────────────
# Image counts seen when these datasets were copied successfully in July (SSL_SR notebook).
# A different count means Drive returned an incomplete listing or the folder is not the full dataset.
EXPECTED_IMAGES = {'FloraBD': 19426, 'RiceLeaf': 9769, 'PlantVillage': 44014,
                   'Flowers102': 8189, 'CUB': 11788}
# If a dataset lives elsewhere on Drive, point to it here before calling localize(), e.g.
#   S.SOURCE_OVERRIDE['FloraBD'] = '/content/drive/MyDrive/FloraBD/FloraBD_224_split'
_SRC_FILE = f'{BASE}/results_v2/data_sources.json'      # shared by all notebooks
SOURCE_OVERRIDE = json.load(open(_SRC_FILE)) if os.path.exists(_SRC_FILE) else {}
ALLOW_COUNT_MISMATCH = False


def set_source(key, path):
    """Use a different Drive folder for a dataset; saved so every notebook uses it."""
    assert key in DATASETS, key
    assert os.path.isdir(path), f'not a folder: {path}'
    SOURCE_OVERRIDE[key] = path
    os.makedirs(os.path.dirname(_SRC_FILE), exist_ok=True)
    json.dump(SOURCE_OVERRIDE, open(_SRC_FILE, 'w'), indent=1)
    print(f'{key} -> {path}  (saved to {_SRC_FILE})')


def source_dir(key):
    return SOURCE_OVERRIDE.get(key, f'{BASE}/datasets/{DATASETS[key]}')


def _count_imgs(p):
    if not os.path.isdir(p):
        return 0
    return sum(1 for _, _, fs in os.walk(p, followlinks=True)
               for f in fs if f.lower().endswith(IMG_EXT))


def _scan(src, retries=3):
    """One walk of the source (follows Drive shortcuts). Retries because the
    Drive FUSE mount can return empty or partial listings right after mounting."""
    best = []
    for attempt in range(retries):
        files = [os.path.relpath(os.path.join(r, f), src)
                 for r, _, fs in os.walk(src, followlinks=True)
                 for f in fs if f.lower().endswith(IMG_EXT)]
        if len(files) > len(best):
            best = files
        if best and len(files) == len(best) and attempt > 0:
            break                       # two identical listings in a row -> stable
        time.sleep(3)
    return sorted(best)


def describe_source(key):
    """Per-split image counts of the Drive source folder (diagnostics)."""
    src = source_dir(key)
    print(f'{key}: {src}')
    if not os.path.isdir(src):
        print('   folder does not exist'); return
    for e in sorted(os.listdir(src)):
        p = os.path.join(src, e)
        kind = 'shortcut/symlink' if os.path.islink(p) else 'dir' if os.path.isdir(p) else 'file'
        n = _count_imgs(p) if os.path.isdir(p) else ''
        print(f'   {e:20s} {kind:16s} {n}')


TAR_CACHE = f'{BASE}/datasets_tar'     # one uncompressed .tar per dataset (built once, reused by every session)


def _local_count(dst):
    return _count_imgs(dst)


def _split_counts(paths):
    c = {}
    for r in paths:
        k = r.split(os.sep)[0]
        c[k] = c.get(k, 0) + 1
    return c


def _check_count(key, n, src):
    exp = EXPECTED_IMAGES.get(key)
    if (exp and n != exp and not ALLOW_COUNT_MISMATCH and os.environ.get('SSL_SMOKE') != '1'):
        describe_source(key)
        raise RuntimeError(
            f'{key}: found {n} images in {src}, expected {exp}.\n'
            f'Either Drive has not finished listing the folder (re-run the cell), or this is not the '
            f'full dataset. Point to the right folder with S.set_source("{key}", "..."), or set '
            f'S.ALLOW_COUNT_MISMATCH = True only if you are sure the new count is correct.')


def _from_tar(key, src, dst):
    """Fast path: copy ONE large file from Drive and extract locally (seconds-minutes
    instead of tens of thousands of small Drive reads). Returns True on success."""
    tar_f, man_f = f'{TAR_CACHE}/{key}.tar', f'{TAR_CACHE}/{key}.json'
    if not (os.path.exists(tar_f) and os.path.exists(man_f)):
        return False
    man = json.load(open(man_f))
    if man.get('source') != src:
        print(f'  [{key}] tar cache was built from a different folder — rebuilding'); return False
    t0 = time.time()
    tmp = f'/tmp/_{key}.tar'
    shutil.copyfile(tar_f, tmp)
    t1 = time.time()
    os.makedirs(dst, exist_ok=True)
    subprocess.run(['tar', '-xf', tmp, '-C', dst], check=True)
    os.remove(tmp)
    n = _local_count(dst)
    if n != man['n']:
        print(f'  [{key}] tar extract gave {n} images, manifest says {man["n"]} — falling back to file copy')
        shutil.rmtree(dst, ignore_errors=True); return False
    gb = os.path.getsize(tar_f) / 1e9
    print(f'  [{key}] ✓ {n} images local from tar cache ({gb:.2f} GB: copy {t1-t0:.0f}s, '
          f'extract {time.time()-t1:.0f}s)  {man["splits"]}')
    return True


def _build_tar(key, src, dst, files):
    """After a file-by-file copy, pack the LOCAL copy into one tar on Drive for next time."""
    try:
        os.makedirs(TAR_CACHE, exist_ok=True)
        tmp = f'/tmp/_{key}_build.tar'
        lst = f'/tmp/_{key}_files.txt'
        open(lst, 'w').write('\n'.join(files) + '\n')
        subprocess.run(['tar', '-cf', tmp, '-C', dst, '-T', lst], check=True)
        tar_f = f'{TAR_CACHE}/{key}.tar'
        shutil.copyfile(tmp, tar_f + '.part')
        os.replace(tar_f + '.part', tar_f)
        json.dump({'source': src, 'n': len(files), 'splits': _split_counts(files),
                   'built': time.strftime('%Y-%m-%d %H:%M:%S')},
                  open(f'{TAR_CACHE}/{key}.json', 'w'), indent=1)
        os.remove(tmp); os.remove(lst)
        print(f'  [{key}] tar cache written to Drive ({os.path.getsize(tar_f)/1e9:.2f} GB) — '
              f'future sessions will load it in one copy')
    except Exception as e:                        # cache is an optimisation only
        print(f'  [{key}] ⚠ could not write tar cache ({e}); next session will copy file-by-file again')


def localize(names=None, workers=64):
    """
    Put datasets on local disk. Order of preference:
      1. already local in this VM  -> nothing to do
      2. tar cache on Drive        -> copy one big file + extract (fast)
      3. file-by-file parallel copy from Drive (slow, first time only), then
         build the tar cache so every later session uses path 2.
    Image counts are always checked against the expected totals.
    """
    from tqdm.auto import tqdm
    names = names or list(DATASETS)
    os.makedirs(LOCAL, exist_ok=True)
    for key in names:
        src, dst = source_dir(key), f'{LOCAL}/{key}'
        exp = EXPECTED_IMAGES.get(key)
        n_local = _local_count(dst)
        if n_local and (exp is None or n_local == exp or os.environ.get('SSL_SMOKE') == '1'):
            print(f'  [{key}] ✓ already local ({n_local} images)')
            if not os.path.exists(f'{TAR_CACHE}/{key}.tar') and os.path.isdir(src):
                files = sorted(os.path.relpath(os.path.join(r, f), dst)
                               for r, _, fs in os.walk(dst) for f in fs if f.lower().endswith(IMG_EXT))
                _build_tar(key, src, dst, files)
            continue
        if _from_tar(key, src, dst):
            _check_count(key, _local_count(dst), src)
            continue
        if not os.path.isdir(src):
            raise FileNotFoundError(f'{key}: source folder missing: {src}')
        files = _scan(src)
        _check_count(key, len(files), src)
        todo = [r for r in files if not os.path.exists(os.path.join(dst, r))]
        for d in {os.path.dirname(os.path.join(dst, r)) for r in todo}:
            os.makedirs(d, exist_ok=True)
        if todo:
            def _cp(r):
                shutil.copyfile(os.path.join(src, r), os.path.join(dst, r))
                return 1
            with ThreadPoolExecutor(workers) as ex:
                list(tqdm(ex.map(_cp, todo), total=len(todo), desc=f'{key} (first-time copy)',
                          unit='img', mininterval=2))
        missing = [r for r in files if not os.path.exists(os.path.join(dst, r))]
        assert not missing, f'{key}: {len(missing)} files failed to copy, e.g. {missing[:3]}'
        print(f'  [{key}] ✓ {len(files)} images local  {_split_counts(files)}')
        _build_tar(key, src, dst, files)


# ───────────────────────── splits ─────────────────────────
def _list_folder(folder):
    """Return {class_name: [sorted image paths]} for non-empty class dirs."""
    out = {}
    for c in sorted(os.listdir(folder)):
        p = os.path.join(folder, c)
        if os.path.isdir(p):
            imgs = sorted(os.path.join(p, f) for f in os.listdir(p)
                          if f.lower().endswith(IMG_EXT))
            if imgs:
                out[c] = imgs
    return out


def get_splits(key, verbose=True):
    """
    Returns dict with 'classes' and 'train'/'val'/'test' lists of (path, label).
    Class indices come from TRAIN; val/test images of classes absent from train
    are excluded and reported (never silently re-indexed).
    """
    root = f'{LOCAL}/{key}'
    tr = _list_folder(f'{root}/train')
    classes = sorted(tr)
    cidx = {c: i for i, c in enumerate(classes)}

    def to_samples(d, name):
        s, dropped = [], 0
        for c, imgs in d.items():
            if c in cidx:
                s += [(p, cidx[c]) for p in imgs]
            else:
                dropped += len(imgs)
        if dropped:
            print(f'  ⚠ {key}/{name}: {dropped} images in classes not present in train — excluded')
        return s

    if key in VAL_IS_TEST:
        test = to_samples(_list_folder(f'{root}/val'), 'val(=official test)')
        split_f = f'{SPLITS}/{key}_carved_val.json'
        def _rel(p):            # 'train/<class>/<file>' — independent of where LOCAL is
            p = p.replace('\\', '/')
            i = p.find(f'/{key}/train/')
            return p[i + len(key) + 2:] if i >= 0 else p
        if os.path.exists(split_f):
            val_rel = {_rel(p) for p in json.load(open(split_f))['val_paths']}
        else:
            rng = random.Random(SPLIT_SEED)
            val_rel = set()
            for c, imgs in tr.items():
                imgs = list(imgs); rng.shuffle(imgs)
                n = max(1, int(round(len(imgs) * CARVE_VAL_FRAC)))
                val_rel.update(_rel(p) for p in imgs[:n])
            os.makedirs(SPLITS, exist_ok=True)
            json.dump({'seed': SPLIT_SEED, 'frac': CARVE_VAL_FRAC,
                       'val_paths': sorted(val_rel)}, open(split_f, 'w'))
        val_paths = {p for p, _ in to_samples(tr, 'train') if _rel(p) in val_rel}
        if len(val_paths) != len(val_rel):
            raise RuntimeError(f'{key}: carved val split lists {len(val_rel)} images but only '
                               f'{len(val_paths)} were found locally — dataset copy incomplete?')
        all_tr = to_samples(tr, 'train')
        train = [s for s in all_tr if s[0] not in val_paths]
        val = [s for s in all_tr if s[0] in val_paths]
    else:
        for need in ('val', 'test'):
            if not os.path.isdir(f'{root}/{need}'):
                raise FileNotFoundError(f'{key}: expected {root}/{need}')
        train = to_samples(tr, 'train')
        val = to_samples(_list_folder(f'{root}/val'), 'val')
        test = to_samples(_list_folder(f'{root}/test'), 'test')

    # guard against overlap between splits (same file path)
    a, b, c = {p for p, _ in train}, {p for p, _ in val}, {p for p, _ in test}
    assert not (a & b) and not (a & c) and not (b & c), f'{key}: split overlap'
    if verbose:
        print(f'  {key:12s} classes={len(classes):3d}  train={len(train):6d}  '
              f'val={len(val):5d}  test={len(test):5d}'
              + ('   (val carved from train; test = official test)' if key in VAL_IS_TEST else ''))
    return {'classes': classes, 'train': train, 'val': val, 'test': test}


def split_fingerprint(samples):
    h = hashlib.md5()
    for p, y in samples:
        h.update(f'{os.path.relpath(p, LOCAL)}|{y}\n'.encode())
    return h.hexdigest()


class ListDataset(Dataset):
    """(path, label) list -> (img, label, ok). Unreadable images get ok=False."""
    def __init__(self, samples, tf):
        self.samples, self.tf = samples, tf
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        p, y = self.samples[i]
        try:
            img = Image.open(p).convert('RGB')
            return self.tf(img), y, True
        except Exception:
            return torch.zeros(3, 224, 224), y, False


# ───────────────────────── backbones ─────────────────────────
def _vits():
    if DINO_REPO not in sys.path:
        sys.path.insert(0, DINO_REPO)
    import vision_transformer as vits
    return vits


def _strip_dino(sd):
    for k in ('teacher', 'student', 'model', 'state_dict'):
        if isinstance(sd, dict) and k in sd and isinstance(sd[k], dict):
            sd = sd[k]; break
    out = {}
    for k, v in sd.items():
        k = k.replace('module.', '').replace('backbone.', '')
        if k.startswith('head.') or k.startswith('head'):  # DINO projection head only
            continue
        out[k] = v
    return out


def _load_strict(model, sd, name):
    msg = model.load_state_dict(sd, strict=False)
    if msg.missing_keys or msg.unexpected_keys:
        raise RuntimeError(f'{name}: load mismatch\n missing={msg.missing_keys[:10]}'
                           f'\n unexpected={msg.unexpected_keys[:10]}')
    n = len(model.state_dict())
    print(f'  ✓ {name}: {n}/{n} backbone tensors loaded (strict)')
    return model


DINO_IN_URL = ('https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/'
               'dino_deitsmall16_pretrain.pth')
MAE_URL = 'https://dl.fbaipublicfiles.com/mae/pretrain/mae_pretrain_vit_base.pth'

# Domain checkpoints: name -> path. Missing files are skipped by callers.
def domain_checkpoints():
    d = {}
    for n in ['florabd_1k', 'florabd_5k', 'florabd_10k', 'rice_1k', 'rice_5k',
              'rice_full', 'pv_5k', 'pv_10k', 'pv_full']:
        d[n] = f'{CKPT_DOM}/{n}/checkpoint.pth'
    for n in ['florabd_full_scratch', 'florabd_cont_in', 'rice_cont_in'] + REVISION_RUNS:
        d[n] = f'{CKPT_DOM2}/{n}/checkpoint.pth'
    return d


# Revision runs (notebooks 08-10). Used only once their COMPLETE marker exists.
REVISION_RUNS = [
    # 08: generic-data control, from scratch, image counts and steps matched to the RiceLeaf arms
    'inet_n996', 'inet_n4671', 'inet_n6835', 'inet_n30829',
    # 09: extra pretraining seeds for the key arms
    'rice_full_s1', 'rice_full_s2', 'rice_cont_in_s1', 'rice_cont_in_s2',
    # 10: training budget / recipe
    'rice_full_ep400', 'rice_full_inrecipe', 'rice_1k_stepmatch', 'rice_5k_stepmatch',
    # 10: continued-pretraining variants
    'rice_cont_fullhead', 'rice_cont_fullhead_lowlr', 'rice_cont_replay',
]

# pretraining images per checkpoint (for scaling plots)
PRETRAIN_IMAGES = {
    'imagenet_dino': 1281167, 'florabd_1k': 990, 'florabd_5k': 4995,
    'florabd_10k': 9671, 'florabd_full_scratch': 13598, 'rice_1k': 996,
    'rice_5k': 4671, 'rice_full': 6835, 'pv_5k': 4920, 'pv_10k': 9444,
    'pv_full': 30829, 'florabd_cont_in': 1281167 + 13598,
    'rice_cont_in': 1281167 + 6835,
    'inet_n996': 996, 'inet_n4671': 4671, 'inet_n6835': 6835, 'inet_n30829': 30829,
    'rice_full_s1': 6835, 'rice_full_s2': 6835, 'rice_cont_in_s1': 1281167 + 6835,
    'rice_cont_in_s2': 1281167 + 6835, 'rice_full_ep400': 6835, 'rice_full_inrecipe': 6835,
    'rice_1k_stepmatch': 996, 'rice_5k_stepmatch': 4671, 'rice_cont_fullhead': 1281167 + 6835,
    'rice_cont_fullhead_lowlr': 1281167 + 6835, 'rice_cont_replay': 1281167 + 6835,
}


def is_complete(name):
    """Domain checkpoint usable? The 9 July runs are complete; notebook-02 runs only
    once notebook 02 has written its COMPLETE marker (DINO writes checkpoint.pth
    every epoch, so file existence alone would expose half-trained models)."""
    p = domain_checkpoints()[name]
    if not os.path.exists(p):
        return False
    if p.startswith(CKPT_DOM2):
        return os.path.exists(os.path.join(os.path.dirname(p), 'COMPLETE'))
    return True


def backbone_id(name):
    """Identity of the exact weights behind a backbone name. For domain checkpoints
    it changes whenever checkpoint.pth is rewritten (e.g. DINO resumed training),
    so results and feature caches from an older/incomplete checkpoint are rejected."""
    if name in domain_checkpoints():
        p = domain_checkpoints()[name]
        if not os.path.exists(p):
            return f'{name}:missing'
        st = os.stat(p)
        return f'{name}:{st.st_size}:{int(st.st_mtime)}'
    return name


def _is_dom2(name):
    return name in domain_checkpoints() and domain_checkpoints()[name].startswith(CKPT_DOM2)


def result_valid(rec, name):
    """A saved result counts as done only if it came from the current weights.
    Records written before ckpt_id existed are trusted only for checkpoints that
    can no longer change (ImageNet/DINOv2/MAE and the 9 finished July domain runs)."""
    if 'ckpt_id' in rec:
        return rec['ckpt_id'] == backbone_id(name)
    return not _is_dom2(name)


def available_backbones(include_external=True):
    names = ['imagenet_dino'] + [n for n in domain_checkpoints() if is_complete(n)]
    if include_external:
        names += ['dinov2_vits14', 'dinov2_vitb14', 'mae_vitb16']
    return names


def load_backbone(name):
    """Return (model in eval mode, feature_dim). Raises on any load mismatch."""
    if name == 'imagenet_dino':
        vits = _vits()
        m = vits.vit_small(patch_size=16, num_classes=0)
        f = f'{CKPT_PRE}/dino_deitsmall16_pretrain.pth'
        if not os.path.exists(f):
            os.makedirs(CKPT_PRE, exist_ok=True)
            torch.hub.download_url_to_file(DINO_IN_URL, f)
        sd = torch.load(f, map_location='cpu', weights_only=False)
        return _load_strict(m, _strip_dino(sd), name).eval(), 384
    if name in domain_checkpoints():
        if not is_complete(name):
            raise RuntimeError(f'{name}: checkpoint missing or pretraining not finished (notebook 02).')
        vits = _vits()
        m = vits.vit_small(patch_size=16, num_classes=0)
        ck = torch.load(domain_checkpoints()[name], map_location='cpu', weights_only=False)
        ep = ck.get('epoch', None) if isinstance(ck, dict) else None
        if ep is not None:
            print(f'  {name}: checkpoint epoch = {ep}')
        return _load_strict(m, _strip_dino(ck), name).eval(), 384
    if name == 'dinov2_vits14':
        return torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').eval(), 384
    if name == 'dinov2_vitb14':
        return torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14').eval(), 768
    if name == 'mae_vitb16':
        import timm
        m = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=0)
        f = f'{CKPT_PRE}/mae_vitbase16.pth'
        if not os.path.exists(f):
            os.makedirs(CKPT_PRE, exist_ok=True)
            torch.hub.download_url_to_file(MAE_URL, f)
        sd = torch.load(f, map_location='cpu', weights_only=False)
        sd = sd.get('model', sd)
        sd = {k: v for k, v in sd.items()
              if not k.startswith('decoder') and k != 'mask_token'}
        return _load_strict(m, sd, name).eval(), 768
    raise KeyError(name)


def _forward_feats(model, x):
    out = model(x)
    if isinstance(out, dict):
        out = out['x_norm_clstoken']
    elif isinstance(out, (list, tuple)):
        out = out[0]
    return out


# ───────────────────────── feature cache ─────────────────────────
@torch.no_grad()
def extract(model, samples, batch=256, workers=None):
    workers = NUM_WORKERS if workers is None else workers
    dl = DataLoader(ListDataset(samples, EVAL_TF), batch_size=batch, shuffle=False,
                    num_workers=workers, pin_memory=torch.cuda.is_available())
    model = model.to(DEVICE).eval()
    X, Y, OK = [], [], []
    for x, y, ok in dl:
        with torch.autocast('cuda', enabled=torch.cuda.is_available()):
            f = _forward_feats(model, x.to(DEVICE, non_blocking=True))
        X.append(f.float().cpu()); Y.append(y); OK.append(ok)
    X, Y, OK = torch.cat(X), torch.cat(Y), torch.cat(OK)
    return X[OK].numpy(), Y[OK].numpy(), int((~OK).sum())


def get_features(backbone_name, key, splits=None, model=None):
    """
    Cached features for train/val/test of dataset `key`.
    Cache is keyed on the exact sample list, so a changed split forces re-extraction.
    """
    splits = splits or get_splits(key, verbose=False)
    out = {}
    for sp in ('train', 'val', 'test'):
        f = f'{FEATS}/{backbone_name}/{key}_{sp}.npz'
        fp = split_fingerprint(splits[sp])
        bid = backbone_id(backbone_name)
        if os.path.exists(f):
            z = np.load(f, allow_pickle=False)
            same_w = (str(z['bid']) == bid) if 'bid' in z.files else not _is_dom2(backbone_name)
            if str(z['fp']) == fp and same_w:
                out[sp] = (z['X'], z['y']); continue
            print(f'  cache stale for {backbone_name}/{key}/{sp} (split or weights changed) — re-extracting')
        if model is None:
            model, _ = load_backbone(backbone_name)
        t0 = time.time()
        X, y, bad = extract(model, splits[sp])
        if bad:
            print(f'  ⚠ {key}/{sp}: {bad} unreadable images skipped')
        os.makedirs(os.path.dirname(f), exist_ok=True)
        tmp = f[:-4] + f'.tmp{os.getpid()}.npz'          # atomic: parallel sessions never read half a file
        np.savez(tmp, X=X.astype(np.float32), y=y, fp=fp, bad=bad, bid=bid)
        os.replace(tmp, f)
        print(f'  cached {backbone_name}/{key}/{sp}: {X.shape} in {time.time()-t0:.0f}s')
        out[sp] = (X, y)
    return out, model


# ───────────────────────── linear probe ─────────────────────────
C_GRID = [0.01, 0.0316, 0.1, 0.316, 1.0, 3.16]


def linear_probe(Xtr, ytr, Xva, yva, Xte, yte, Cs=C_GRID, refit_trainval=True):
    """
    Standardize on train, choose C on val, refit on train+val, score test once.
    Returns dict(test_acc, val_acc, C, correct[bool array over test]).
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    sc = StandardScaler().fit(Xtr)
    best = (-1, None)
    for C in Cs:
        clf = LogisticRegression(C=C, max_iter=3000)
        clf.fit(sc.transform(Xtr), ytr)
        acc = clf.score(sc.transform(Xva), yva)
        if acc > best[0]:
            best = (acc, C)
    val_acc, C = best
    if refit_trainval:
        Xf, yf = np.concatenate([Xtr, Xva]), np.concatenate([ytr, yva])
        sc = StandardScaler().fit(Xf)
    else:
        Xf, yf = Xtr, ytr
    clf = LogisticRegression(C=C, max_iter=3000).fit(sc.transform(Xf), yf)
    pred = clf.predict(sc.transform(Xte))
    correct = pred == yte
    return {'test_acc': 100 * correct.mean(), 'val_acc': 100 * val_acc,
            'C': C, 'correct': correct}


def stratified_subset(labels, fraction, seed, min_per_class=1):
    """Indices of a class-stratified subset; returns (idx, n_per_class_list)."""
    rng = np.random.RandomState(seed)
    idx, per = [], []
    for c in np.unique(labels):
        ci = np.where(labels == c)[0]
        rng.shuffle(ci)
        n = max(min_per_class, int(round(len(ci) * fraction)))
        n = min(n, len(ci))
        idx.extend(ci[:n].tolist()); per.append(n)
    return np.array(sorted(idx)), per


# ───────────────────────── statistics ─────────────────────────
def bootstrap_ci(correct, n_boot=2000, seed=0, alpha=0.05):
    rng = np.random.RandomState(seed)
    c = np.asarray(correct, dtype=float)
    accs = c[rng.randint(0, len(c), (n_boot, len(c)))].mean(1) * 100
    return float(np.percentile(accs, 100 * alpha / 2)), float(np.percentile(accs, 100 * (1 - alpha / 2)))


def mcnemar(correct_a, correct_b):
    """Exact McNemar test on paired per-image correctness. Returns (b, c, p)."""
    from scipy.stats import binomtest
    a, b = np.asarray(correct_a, bool), np.asarray(correct_b, bool)
    n01, n10 = int((a & ~b).sum()), int((~a & b).sum())
    if n01 + n10 == 0:
        return n01, n10, 1.0
    return n01, n10, float(binomtest(n01, n01 + n10, 0.5).pvalue)


def paired_bootstrap_diff(correct_a, correct_b, n_boot=2000, seed=0):
    rng = np.random.RandomState(seed)
    a, b = np.asarray(correct_a, float), np.asarray(correct_b, float)
    i = rng.randint(0, len(a), (n_boot, len(a)))
    d = (a[i].mean(1) - b[i].mean(1)) * 100
    return float((a.mean() - b.mean()) * 100), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))


def holm(pvals):
    """Holm-Bonferroni adjusted p-values (same order as input)."""
    p = np.asarray(pvals, float); m = len(p)
    order = np.argsort(p); adj = np.empty(m); run = 0
    for r, i in enumerate(order):
        run = max(run, (m - r) * p[i]); adj[i] = min(1.0, run)
    return adj.tolist()


# ───────────────────────── results I/O ─────────────────────────
def _json_default(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.bool_,)): return bool(o)
    raise TypeError(type(o))


def _rdir(fname):
    return f"{RESULTS}/{fname[:-5] if fname.endswith('.json') else fname}_runs"


def load_results(fname):
    """
    Merge of (a) the legacy single JSON file and (b) one small JSON per run in
    <name>_runs/. One-file-per-run means several Colab sessions can write results
    for the same notebook at the same time without overwriting each other.
    """
    data = {}
    f = f'{RESULTS}/{fname}'
    if os.path.exists(f):
        data.update(json.load(open(f)))
    d = _rdir(fname)
    if os.path.isdir(d):
        for g in sorted(os.listdir(d)):
            if g.endswith('.json'):
                try:
                    rec = json.load(open(os.path.join(d, g)))
                    data[rec['__key']] = rec['record']
                except Exception as e:          # half-synced file from another session
                    print(f'  ⚠ skipped unreadable result {g}: {e}')
    return data


def save_result(fname, key, record):
    """One file per run, written atomically (safe for parallel sessions and disconnects)."""
    d = _rdir(fname)
    os.makedirs(d, exist_ok=True)
    safe = hashlib.md5(key.encode()).hexdigest()[:10] + '_' + ''.join(
        c if c.isalnum() or c in '._-' else '_' for c in key)[:150]
    f = os.path.join(d, safe + '.json')
    tmp = f + f'.tmp{os.getpid()}'
    json.dump({'__key': key, 'record': record}, open(tmp, 'w'), indent=1, default=_json_default)
    os.replace(tmp, f)


def save_correct(fname_stub, key, correct):
    """Per-image correctness vectors (needed for McNemar / bootstrap later)."""
    d = f'{RESULTS}/correct/{fname_stub}'
    os.makedirs(d, exist_ok=True)
    np.save(f'{d}/{key}.npy', np.asarray(correct, bool))


def load_correct(fname_stub, key):
    f = f'{RESULTS}/correct/{fname_stub}/{key}.npy'
    return np.load(f) if os.path.exists(f) else None


def env_info():
    info = {'python': platform.python_version(), 'torch': torch.__version__,
            'cuda': torch.cuda.is_available(),
            'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            'time': time.strftime('%Y-%m-%d %H:%M:%S')}
    try:
        cpu = [l for l in open('/proc/cpuinfo') if l.startswith('model name')]
        info['cpu'] = cpu[0].split(':', 1)[1].strip() if cpu else platform.processor()
        info['cpu_threads'] = os.cpu_count()
    except Exception:
        info['cpu'] = platform.processor()
    return info


# ───────────────────────── fine-tuning ─────────────────────────
class _Head(nn.Module):
    def __init__(self, backbone, dim, n_cls):
        super().__init__()
        self.backbone, self.fc = backbone, nn.Linear(dim, n_cls)
    def forward(self, x):
        return self.fc(_forward_feats(self.backbone, x))


@torch.no_grad()
def evaluate(model, loader):
    model.eval(); C = []
    for x, y, ok in loader:
        with torch.autocast('cuda', enabled=torch.cuda.is_available()):
            p = model(x.to(DEVICE, non_blocking=True)).argmax(1).cpu()
        C.append((p == y)[ok])
    C = torch.cat(C).numpy()
    return 100 * C.mean(), C


def make_loaders(splits, batch, workers=None):
    workers = NUM_WORKERS if workers is None else workers
    pin = torch.cuda.is_available()
    tr = DataLoader(ListDataset(splits['train'], TRAIN_TF), batch_size=batch,
                    shuffle=True, num_workers=workers, pin_memory=pin, drop_last=True,
                    persistent_workers=workers > 0)
    va = DataLoader(ListDataset(splits['val'], EVAL_TF), batch_size=256,
                    num_workers=workers, pin_memory=pin)
    te = DataLoader(ListDataset(splits['test'], EVAL_TF), batch_size=256,
                    num_workers=workers, pin_memory=pin)
    return tr, va, te


CKPT_EVERY_MIN = 10      # mid-run checkpoint interval (minutes) for train_classifier


def train_classifier(model, splits, seed, epochs, lr, wd, batch,
                     teacher=None, T=4.0, alpha=0.5, log_every=5, resume_key=None):
    """
    Generic supervised / KD training loop.
    Selects the epoch with best VAL accuracy and reports TEST accuracy of that
    epoch's weights (test is evaluated exactly once).

    resume_key: if given, the full training state (model, optimizer, scheduler,
    scaler, best weights, RNG states) is saved to Drive at most every
    CKPT_EVERY_MIN minutes and after the last epoch; a re-run with the same key
    continues from the last saved epoch instead of epoch 0. The file is deleted
    when the run finishes. A resumed run is statistically equivalent to an
    uninterrupted one but not bit-identical (data-loader worker order differs).
    """
    set_seed(seed)
    tr, va, te = make_loaders(splits, batch)
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
    ce = nn.CrossEntropyLoss(reduction='none')
    kl = nn.KLDivLoss(reduction='none')
    if teacher is not None:
        teacher = teacher.to(DEVICE).eval()
    best_val, best_ep, best_sd = -1, -1, None
    start_ep = 0
    ck_f = f'{RESULTS}/_inprogress/{resume_key}.pt' if resume_key else None
    if ck_f and os.path.exists(ck_f):
        ck = torch.load(ck_f, map_location='cpu', weights_only=False)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        sched.load_state_dict(ck['sched']); scaler.load_state_dict(ck['scaler'])
        best_val, best_ep, best_sd, start_ep = ck['best_val'], ck['best_ep'], ck['best_sd'], ck['epoch']
        torch.set_rng_state(ck['rng_torch']); np.random.set_state(ck['rng_np']); random.setstate(ck['rng_py'])
        if torch.cuda.is_available() and ck.get('rng_cuda') is not None:
            torch.cuda.set_rng_state_all(ck['rng_cuda'])
        print(f'      ↻ resuming {resume_key} from epoch {start_ep}/{epochs} (best_val={best_val:.2f}@{best_ep})')
    t0 = last_save = time.time()
    for ep in range(start_ep, epochs):
        model.train()
        for x, y, ok in tr:
            x, y, ok = x.to(DEVICE, non_blocking=True), y.to(DEVICE), ok.to(DEVICE)
            if not ok.any():
                continue
            with torch.autocast('cuda', enabled=torch.cuda.is_available()):
                logits = model(x)
                loss = ce(logits, y)
                if teacher is not None:
                    with torch.no_grad():
                        t = teacher(x)
                    lkd = kl(torch.log_softmax(logits.float() / T, 1),
                             torch.softmax(t.float() / T, 1)).sum(1) * T * T
                    loss = alpha * lkd + (1 - alpha) * loss
                loss = (loss * ok).sum() / ok.sum()
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
        sched.step()
        v, _ = evaluate(model, va)
        if v > best_val:
            best_val, best_ep = v, ep + 1
            best_sd = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        if (ep + 1) % log_every == 0 or ep == epochs - 1:
            print(f'      ep {ep+1:3d}/{epochs}  val={v:.2f}  best_val={best_val:.2f}@{best_ep}'
                  f'  ({(time.time()-t0)/60:.1f} min)')
        if ck_f and (time.time() - last_save >= CKPT_EVERY_MIN * 60 or ep == epochs - 1):
            os.makedirs(os.path.dirname(ck_f), exist_ok=True)
            torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
                        'sched': sched.state_dict(), 'scaler': scaler.state_dict(),
                        'best_val': best_val, 'best_ep': best_ep, 'best_sd': best_sd,
                        'epoch': ep + 1, 'rng_torch': torch.get_rng_state(),
                        'rng_np': np.random.get_state(), 'rng_py': random.getstate(),
                        'rng_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None},
                       ck_f + '.tmp')
            os.replace(ck_f + '.tmp', ck_f)          # atomic: a disconnect mid-save never corrupts it
            last_save = time.time()
    model.load_state_dict(best_sd)
    test_acc, correct = evaluate(model, te)
    if ck_f and os.path.exists(ck_f):
        os.remove(ck_f)
    return {'val_acc': best_val, 'best_epoch': best_ep, 'test_acc': test_acc,
            'minutes': (time.time() - t0) / 60}, correct, model


def build_vit_classifier(backbone_name, n_cls):
    bb, dim = load_backbone(backbone_name)
    return _Head(bb, dim, n_cls)


def summarize(vals):
    v = np.asarray(vals, float)
    return {'mean': float(v.mean()), 'std': float(v.std(ddof=1)) if len(v) > 1 else 0.0,
            'n': int(len(v)), 'values': v.tolist()}

In [ ]:
# ── Setup (identical in every notebook) ───────────────────────
import os, sys, subprocess, importlib
IN_COLAB = 'google.colab' in sys.modules
SMOKE = os.environ.get('SSL_SMOKE') == '1'      # local test mode only; leave unset in Colab
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'scikit-learn', 'scipy'], check=True)
DINO_REPO = os.environ.get('SSL_DINO_REPO', '/content/dino_vanilla')
if not os.path.isdir(DINO_REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/facebookresearch/dino.git', DINO_REPO], check=True)
DINO_COMMIT = subprocess.run(['git', '-C', DINO_REPO, 'rev-parse', 'HEAD'],
                             capture_output=True, text=True).stdout.strip()
sys.path.insert(0, os.environ.get('SSL_COMMON_DIR', '/content'))
import sslcommon as S
importlib.reload(S)
S.ensure_dirs()
import numpy as np, pandas as pd, torch, json, time, gc
print('DINO repo commit:', DINO_COMMIT)
print(S.env_info())

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind, ttest_rel
OUTD = f'{S.BASE}/paper_v2'
os.makedirs(f'{OUTD}/tables', exist_ok=True); os.makedirs(f'{OUTD}/figures', exist_ok=True)
ORDER_T = ['FloraBD', 'RiceLeaf', 'PlantVillage', 'Flowers102', 'CUB']
NICE = {'imagenet_dino': 'ImageNet DINO', 'dinov2_vits14': 'DINOv2 ViT-S/14', 'dinov2_vitb14': 'DINOv2 ViT-B/14',
        'mae_vitb16': 'MAE ViT-B/16',
        'florabd_1k': 'FloraBD-1k (scratch)', 'florabd_5k': 'FloraBD-5k (scratch)', 'florabd_10k': 'FloraBD-10k (scratch)',
        'florabd_full_scratch': 'FloraBD-full (scratch)',
        'rice_1k': 'RiceLeaf-1k (scratch)', 'rice_5k': 'RiceLeaf-5k (scratch)', 'rice_full': 'RiceLeaf-full (scratch)',
        'pv_5k': 'PlantVillage-5k (scratch)', 'pv_10k': 'PlantVillage-10k (scratch)', 'pv_full': 'PlantVillage-full (scratch)',
        'florabd_cont_in': 'ImageNet→FloraBD (cont.)', 'rice_cont_in': 'ImageNet→RiceLeaf (cont.)',
        'mobilenet_v3_small': 'MobileNetV3-Small', 'efficientnet_b0': 'EfficientNet-B0', 'vit_small_teacher': 'ViT-S/16 (teacher)'}
ROW_ORDER = ['imagenet_dino', 'dinov2_vits14', 'dinov2_vitb14', 'mae_vitb16',
             'florabd_1k', 'florabd_5k', 'florabd_10k', 'florabd_full_scratch',
             'rice_1k', 'rice_5k', 'rice_full', 'pv_5k', 'pv_10k', 'pv_full', 'florabd_cont_in', 'rice_cont_in']
nice = lambda b: NICE.get(b, b)
def ordered(idx): return sorted(idx, key=lambda b: ROW_ORDER.index(b) if b in ROW_ORDER else 99)
def save_table(df, name, caption):
    df.to_csv(f'{OUTD}/tables/{name}.csv')
    with open(f'{OUTD}/tables/{name}.tex', 'w') as f:
        f.write(df.to_latex(escape=True, caption=caption, label=f'tab:{name}'))
    print('✓ table', name); display(df)
def save_fig(name):
    plt.tight_layout()
    for ext in ('png', 'pdf'): plt.savefig(f'{OUTD}/figures/{name}.{ext}', dpi=300)
    plt.close(); print('✓ figure', name)
def have(f): return os.path.exists(f'{S.RESULTS}/{f}') or bool(S.load_results(f))
def stars(p): return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''

In [ ]:
# Table: datasets
if have('00_splits.json'):
    sp = pd.DataFrame(json.load(open(f'{S.RESULTS}/00_splits.json'))).T
    save_table(sp[['classes', 'train', 'val', 'test', 'val_carved_from_train']], 'tab_datasets', 'Datasets and splits.')

In [ ]:
# Tables + figure: frozen linear probe, and difference vs ImageNet DINO (McNemar, Holm over all comparisons)
if have('01_linear_probe.json'):
    lp = pd.DataFrame(S.load_results('01_linear_probe.json').values())
    lp['cell'] = lp.apply(lambda r: f"{r.test_acc:.2f} [{r.ci95[0]:.1f}, {r.ci95[1]:.1f}]", axis=1)
    tab = lp.pivot(index='backbone', columns='target', values='cell')
    tab = tab.reindex(index=ordered(tab.index), columns=[t for t in ORDER_T if t in tab.columns])
    tab.index = [nice(b) for b in tab.index]
    save_table(tab, 'tab_linear_probe', 'Frozen linear-probe test accuracy (\\%) with 95\\% bootstrap CI.')
if have('01_lp_vs_imagenet.json'):
    cmp = pd.DataFrame(json.load(open(f'{S.RESULTS}/01_lp_vs_imagenet.json')))
    cmp['cell'] = cmp.apply(lambda r: f"{r.diff_vs_imagenet:+.2f}{stars(r.p_holm)}", axis=1)
    dom = cmp[~cmp.backbone.isin(['dinov2_vits14', 'dinov2_vitb14', 'mae_vitb16'])]
    tab = dom.pivot(index='backbone', columns='target', values='cell')
    tab = tab.reindex(index=ordered(tab.index), columns=[t for t in ORDER_T if t in tab.columns])
    tab.index = [nice(b) for b in tab.index]
    save_table(tab, 'tab_domain_vs_imagenet', 'Linear-probe accuracy difference vs ImageNet DINO (pp). '
               'McNemar test, Holm-corrected over all %d comparisons: * p<0.05, ** p<0.01, *** p<0.001.' % len(cmp))
    piv = dom.pivot(index='backbone', columns='target', values='diff_vs_imagenet')
    piv = piv.reindex(index=ordered(piv.index), columns=[t for t in ORDER_T if t in piv.columns])
    fig, ax = plt.subplots(figsize=(1.3 * piv.shape[1] + 3.5, 0.42 * piv.shape[0] + 1.5))
    v = np.nanmax(np.abs(piv.values)) or 1
    im = ax.imshow(piv.values, cmap='RdBu', vmin=-v, vmax=v, aspect='auto')
    ax.set_xticks(range(piv.shape[1]), piv.columns); ax.set_yticks(range(piv.shape[0]), [nice(b) for b in piv.index])
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            if not np.isnan(piv.values[i, j]):
                ax.text(j, i, f'{piv.values[i, j]:+.1f}', ha='center', va='center', fontsize=8)
    plt.colorbar(im, label='Δ accuracy vs ImageNet DINO (pp)')
    save_fig('fig_domain_vs_imagenet')

In [ ]:
# Figure + table: scaling of from-scratch domain DINO (linear probe) and an indicative log-linear fit
if have('01_linear_probe.json'):
    lp = pd.DataFrame(S.load_results('01_linear_probe.json').values())
    src = lambda b: ('FloraBD' if b.startswith('florabd') else 'RiceLeaf' if b.startswith('rice')
                     else 'PlantVillage' if b.startswith('pv') else None)
    lp['source'] = lp.backbone.map(src); lp['n_img'] = lp.backbone.map(S.PRETRAIN_IMAGES)
    scr = lp[lp.source.notna() & ~lp.backbone.str.contains('cont')]
    targets = [t for t in ORDER_T if t in lp.target.unique()]
    fig, axes = plt.subplots(1, len(targets), figsize=(3.2 * len(targets), 3.0)); axes = np.atleast_1d(axes)
    fits = []
    for ax, t in zip(axes, targets):
        ref = lp[(lp.backbone == 'imagenet_dino') & (lp.target == t)].test_acc
        ref = float(ref.iloc[0]) if len(ref) else np.nan
        for s, g in scr[scr.target == t].groupby('source'):
            g = g.sort_values('n_img')
            ax.plot(g.n_img, g.test_acc, 'o-', lw=2.2 if s == t else 1.2, label=f'{s} (scratch)' + (' · in-domain' if s == t else ''))
            if len(g) >= 3:
                x, y = np.log10(g.n_img.values), g.test_acc.values
                slope, icpt = np.polyfit(x, y, 1)
                r2 = 1 - ((y - (slope * x + icpt)) ** 2).sum() / ((y - y.mean()) ** 2).sum()
                if s != t:          # extrapolation is only meaningful for in-domain curves
                    continue
                n_match = 10 ** ((ref - icpt) / slope) if slope > 0 and ref > y.max() else np.nan
                if np.isfinite(n_match) and n_match > 100 * g.n_img.max():
                    n_match = np.inf
                fits.append({'target': t, 'pretraining source': s, 'in-domain': s == t, 'points': len(g),
                             'images range': f'{int(g.n_img.min()):,}–{int(g.n_img.max()):,}',
                             'pp per 10× images': round(slope, 2), 'R²': round(r2, 3),
                             'ImageNet DINO LP': round(ref, 2),
                             'extrapolated images to match (indicative)': (
                                 'saturated target, not extrapolated' if ref >= 99 else
                                 'already within observed range' if ref <= y.max() else
                                 '>100× observed range' if n_match == np.inf else
                                 f'~{n_match:,.0f} (×{n_match / g.n_img.max():.0f} the largest run)' if np.isfinite(n_match) else 'n/a')})
        if np.isfinite(ref): ax.axhline(ref, ls='--', c='k', lw=1, label='ImageNet DINO')
        ax.set_xscale('log'); ax.set_title(t); ax.set_xlabel('pretraining images'); ax.grid(alpha=.3)
    axes[0].set_ylabel('linear-probe test acc (%)'); axes[-1].legend(fontsize=6)
    save_fig('fig_scaling')
    if fits:
        save_table(pd.DataFrame(fits).set_index(['target', 'pretraining source']), 'tab_scaling_fit',
                   'In-domain log-linear fit of linear-probe accuracy vs number of pretraining images (from-scratch DINO). '
                   'The extrapolation beyond the observed range is indicative only.')

In [ ]:
# Label efficiency: images-per-class axis, duplicate budgets removed, paired tests over subsets (Holm)
if have('03_label_efficiency.json'):
    le = pd.DataFrame(S.load_results('03_label_efficiency.json').values())
    dup = set()
    for t, g in le[le.backbone == 'imagenet_dino'].groupby('target'):
        seen = {}
        for frac in sorted(g.fraction.unique()):
            sig = tuple(g[g.fraction == frac].sort_values('seed').n_train)
            if sig in seen: dup.add((t, frac)); print(f'{t}: {frac:.0%} identical to {seen[sig]:.0%} -> reported once')
            else: seen[sig] = frac
    le = le[[(t, f) not in dup for t, f in zip(le.target, le.fraction)]]
    s = le.groupby(['target', 'fraction', 'backbone']).agg(m=('test_acc', 'mean'), sd=('test_acc', 'std'),
            ipc=('median_per_class', 'first'), n=('test_acc', 'size')).reset_index()
    s['cell'] = s.apply(lambda r: f"{r.m:.2f} ± {r.sd:.2f}" if r.n > 1 else f"{r.m:.2f}", axis=1)
    s['labels'] = s.apply(lambda r: f"{r.fraction:.0%} (~{r.ipc:.0f}/class)", axis=1)
    tab = s.pivot_table(index=['target', 'labels'], columns='backbone', values='cell', aggfunc='first')
    tab = tab[ordered(tab.columns)]; tab.columns = [nice(c) for c in tab.columns]
    tab = tab.reindex(sorted(tab.index, key=lambda k: (ORDER_T.index(k[0]),
                      s[(s.target == k[0]) & (s.labels == k[1])].fraction.iloc[0])))
    n_sub = int(le[le.fraction < 1].groupby(['backbone', 'target', 'fraction']).size().max())
    save_table(tab, 'tab_label_efficiency', f'Label efficiency: linear-probe test accuracy (\\%), mean ± std over {n_sub} '
               'random class-stratified label subsets (fixed C=0.316, trained on the subset only). '
               'Budgets that give identical subsets are reported once.')
    targets = [t for t in ORDER_T if t in s.target.unique()]
    fig, axes = plt.subplots(1, len(targets), figsize=(3.4 * len(targets), 3.1)); axes = np.atleast_1d(axes)
    for ax, t in zip(axes, targets):
        for b in ordered(s[s.target == t].backbone.unique()):
            g = s[(s.target == t) & (s.backbone == b)].sort_values('ipc'); sd = g.sd.fillna(0)
            ax.plot(g.ipc, g.m, 'o-', ms=3, label=nice(b)); ax.fill_between(g.ipc, g.m - sd, g.m + sd, alpha=.15)
        ax.set_xscale('log'); ax.set_title(t); ax.set_xlabel('labelled images per class'); ax.grid(alpha=.3)
    axes[0].set_ylabel('test acc (%)'); axes[-1].legend(fontsize=6)
    save_fig('fig_label_efficiency')
    rows = []
    for (t, frac), g in le[le.fraction < 1].groupby(['target', 'fraction']):
        ref = g[g.backbone == 'imagenet_dino'].set_index('seed').test_acc
        for b in g.backbone.unique():
            if b == 'imagenet_dino': continue
            cur = g[g.backbone == b].set_index('seed').test_acc; c = ref.index.intersection(cur.index)
            if len(c) < 3: continue
            p = ttest_rel(cur[c], ref[c]).pvalue; p = 1.0 if np.isnan(p) else float(p)
            rows.append({'target': t, 'labels': f"{frac:.0%} (~{g.median_per_class.iloc[0]:.0f}/class)", 'backbone': b,
                         'diff_vs_imagenet': (cur[c] - ref[c]).mean(), 'sd_diff': (cur[c] - ref[c]).std(ddof=1),
                         'n_subsets': len(c), 'p': p})
    for r, pa in zip(rows, S.holm([r['p'] for r in rows])): r['p_holm'] = pa
    json.dump(rows, open(f'{S.RESULTS}/07_label_efficiency_tests_dedup.json', 'w'), indent=1, default=S._json_default)
    T = pd.DataFrame(rows)
    T['cell'] = T.apply(lambda r: f"{r.diff_vs_imagenet:+.2f}{stars(r.p_holm)}", axis=1)
    T['frac'] = T['labels'].str.extract(r'^([\d.]+)%')[0].astype(float)
    tt = T.pivot_table(index=['target', 'labels'], columns='backbone', values='cell', aggfunc='first')
    tt = tt.reindex(sorted(tt.index, key=lambda k: (ORDER_T.index(k[0]),
                    T[(T.target == k[0]) & (T['labels'] == k[1])].frac.iloc[0])))
    tt = tt[ordered(tt.columns)]; tt.columns = [nice(c) for c in tt.columns]
    save_table(tt, 'tab_label_efficiency_tests', f'Label efficiency: difference vs ImageNet DINO (pp), paired t-test over '
               f'{n_sub} shared label subsets, Holm-corrected over {len(T)} tests: * p<0.05, ** p<0.01, *** p<0.001.')
    domT = T[~T.backbone.str.startswith('dinov2')]
    print(f'Domain checkpoints vs ImageNet DINO: {((domT.p_holm < .05) & (domT.diff_vs_imagenet > 0)).sum()} significantly better, '
          f'{((domT.p_holm < .05) & (domT.diff_vs_imagenet < 0)).sum()} significantly worse, of {len(domT)} tests')

In [ ]:
# Fine-tuning: mean ± std over seeds, Welch t-test vs ImageNet DINO (Holm)
if have('04_finetune.json'):
    ft = pd.DataFrame(S.load_results('04_finetune.json').values())
    g = ft.groupby(['backbone', 'target']).test_acc.agg(['mean', 'std', 'count']).reset_index()
    g['cell'] = g.apply(lambda r: f"{r['mean']:.2f} ± {r['std']:.2f}" if r['count'] > 1 else f"{r['mean']:.2f}†", axis=1)
    tab = g.pivot(index='backbone', columns='target', values='cell')
    tab = tab.reindex(index=ordered(tab.index), columns=[t for t in ORDER_T if t in tab.columns]); tab.index = [nice(b) for b in tab.index]
    save_table(tab, 'tab_finetune', 'Full fine-tuning test accuracy (\\%), mean ± std over 3 seeds; † single seed (saturated dataset).')
    rows = []
    for t, gt in ft.groupby('target'):
        ref = gt[gt.backbone == 'imagenet_dino'].test_acc.values
        if len(ref) < 2: continue
        for b, gb in gt.groupby('backbone'):
            if b == 'imagenet_dino' or len(gb) < 2: continue
            rows.append({'target': t, 'backbone': b, 'diff_vs_imagenet': gb.test_acc.mean() - ref.mean(),
                         'p_welch': float(ttest_ind(gb.test_acc.values, ref, equal_var=False).pvalue),
                         'seed_values': gb.sort_values('seed').test_acc.round(2).tolist()})
    for r, pa in zip(rows, S.holm([r['p_welch'] for r in rows])): r['p_holm'] = pa
    json.dump(rows, open(f'{S.RESULTS}/07_finetune_tests.json', 'w'), indent=1, default=S._json_default)
    T = pd.DataFrame(rows)
    T['cell'] = T.apply(lambda r: f"{r.diff_vs_imagenet:+.2f}{stars(r.p_holm)}", axis=1)
    tt = T.pivot(index='backbone', columns='target', values='cell')
    tt = tt.reindex(index=ordered(tt.index), columns=[t for t in ORDER_T if t in tt.columns]); tt.index = [nice(b) for b in tt.index]
    save_table(tt, 'tab_finetune_tests', f'Fine-tuning: difference vs ImageNet DINO (pp), Welch t-test over 3 seeds, '
               f'Holm-corrected over {len(T)} tests (low power with n=3): * p<0.05, ** p<0.01, *** p<0.001.')

In [ ]:
# Distillation: students vs plain fine-tuning (Welch, Holm) + teacher accuracies
if have('06_kd.json'):
    allkd = S.load_results('06_kd.json')
    teach = [dict(v, key=k) for k, v in allkd.items() if k.startswith('teacher')]
    if teach:
        tt = pd.DataFrame(teach)
        tt['teacher'] = tt['teacher'].map(nice)
        save_table(tt[['teacher', 'target', 'test_acc', 'val_acc', 'best_epoch']].round(2).set_index(['target', 'teacher']),
                   'tab_kd_teachers', 'ViT-S/16 teachers fine-tuned on the target (seed 0): test accuracy (\\%).')
    else:
        print('⚠ no teacher records found in 06_kd results — teacher accuracies table skipped')
    kd = pd.DataFrame([v for k, v in allkd.items() if not k.startswith('teacher')])
    kd['teacher'] = kd.teacher.fillna('none')
    rows = []
    for (t, st), g in kd.groupby(['target', 'student']):
        base = g[g.method == 'ce'].test_acc.values
        for tb, gk in g[g.method == 'kd'].groupby('teacher'):
            rows.append({'target': t, 'student': nice(st), 'teacher': nice(tb),
                         'CE mean': base.mean(), 'KD mean ± std': f'{gk.test_acc.mean():.2f} ± {gk.test_acc.std():.2f}',
                         'gain_pp': gk.test_acc.mean() - base.mean(),
                         'p_welch': float(ttest_ind(gk.test_acc.values, base, equal_var=False).pvalue)})
    for r, pa in zip(rows, S.holm([r['p_welch'] for r in rows])): r['p_holm'] = pa
    T = pd.DataFrame(rows).round(3)
    save_table(T.set_index(['target', 'student', 'teacher']), 'tab_kd',
               'Logit distillation vs plain fine-tuning of the same student (3 seeds); Welch t-test, Holm-corrected.')

In [ ]:
# Deployment table + Pareto: every student configuration and the ViT-S teacher
if have('06_deploy.json'):
    dep = pd.DataFrame(json.load(open(f'{S.RESULTS}/06_deploy.json')).values())
    cols = ['model', 'target', 'params_M', 'size_MB', 'GMACs', 'cpu_ms_1thread_median', 'cpu_ms_allthreads_median', 'threads']
    d2 = dep[cols].copy(); d2['model'] = d2.model.map(nice)
    save_table(d2.round(3).set_index(['target', 'model']), 'tab_deploy',
               f"Deployment cost on CPU ({dep.env.iloc[0].get('cpu', 'CPU')}), batch 1, fp32 PyTorch, median of 200 runs.")
    if have('06_kd.json'):
        allkd = S.load_results('06_kd.json')
        kd = pd.DataFrame([v for k, v in allkd.items() if not k.startswith('teacher')])
        kd['teacher'] = kd.teacher.fillna('none')
        fig, ax = plt.subplots(figsize=(5.2, 3.6))
        for _, r in dep.iterrows():
            lat = r.cpu_ms_1thread_median
            if r.model == 'vit_small_teacher':
                tv = [v['test_acc'] for k, v in allkd.items() if k.startswith('teacher__imagenet_dino')]
                if tv:
                    ax.scatter(lat, tv[0], marker='*', s=160, c='k'); ax.annotate('ViT-S/16 teacher (ImageNet DINO)', (lat, tv[0]), fontsize=7)
                continue
            g = kd[(kd.student == r.model) & (kd.target == r.target)]
            # plain fine-tune vs the best distillation teacher (other teachers are in tab_kd)
            best_tb = g[g.method == 'kd'].groupby('teacher').test_acc.mean().idxmax() if (g.method == 'kd').any() else None
            for k, (meth, tb, mk, dx) in enumerate([('ce', 'none', 's', 0.97), ('kd', best_tb, 'o', 1.03)]):
                gg = g[(g.method == meth) & (g.teacher == tb)]
                if not len(gg): continue
                lab = f"{nice(r.model)} · {'plain fine-tune' if meth == 'ce' else 'KD ← ' + nice(tb)}"
                ax.errorbar(lat * dx, gg.test_acc.mean(), yerr=gg.test_acc.std(), fmt=mk, ms=5, capsize=2, label=lab)
        ax.set_xscale('log'); ax.set_xlabel('CPU latency, 1 thread (ms, log scale)'); ax.set_ylabel('test acc (%)')
        from matplotlib.ticker import ScalarFormatter
        ax.xaxis.set_major_formatter(ScalarFormatter()); ax.xaxis.set_minor_formatter(ScalarFormatter()); ax.grid(alpha=.3); ax.legend(fontsize=6, loc='lower right')
        save_fig('fig_pareto')
print('\nAll outputs in', OUTD)

In [ ]:
# ── Finish: make sure everything written to Drive is really uploaded ─────
# Colab uploads files to Drive in the background. If the runtime stops (disconnect, units used up,
# closing the tab) before that upload finishes, the most recent results/checkpoints are LOST even
# though the notebook printed them. This cell blocks until the upload is complete.
# Run it at the end of every session and wait for the ✓ before closing the runtime.
if IN_COLAB:
    from google.colab import drive
    drive.flush_and_unmount()
    print('✓ All files uploaded to Drive. You can close this runtime now.')